# Data Science Project

In [14]:
#If not yet installed : 
#pip install kagglehub

In [15]:
#Useful Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

In [16]:
#Preliminary study of the dataset
df = pd.read_csv("athens_data.csv", delimiter=",")

print("Dimensions :")
print(df.shape)

print("\nGeneral Info about dataset")
df.info()

print("\nFirst five values")
print(df.head())

print("\nDescribe dataset")
print(df.describe())

print("\nMissing values")
print(df.isnull().sum())

Dimensions :
(1726464, 19)

General Info about dataset
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1726464 entries, 0 to 1726463
Data columns (total 19 columns):
 #   Column               Dtype  
---  ------               -----  
 0   Date                 object 
 1   Latitude             float64
 2   Longitude            float64
 3   station_name         object 
 4   Wind-Speed (U)       float64
 5   Wind-Speed (V)       float64
 6   Dewpoint Temp        float64
 7   Soil Temp            float64
 8   Total Percipitation  float64
 9   Vegitation (High)    float64
 10  Vegitation (Low)     float64
 11  Temp                 float64
 12  Relative Humidity    float64
 13  PM10                 float64
 14  PM2.5                float64
 15  NO2                  float64
 16  O3                   float64
 17  code                 object 
 18  id                   int64  
dtypes: float64(15), int64(1), object(3)
memory usage: 250.3+ MB

First five values
                  Date  Latitude  

## Exploration of `athens_data.csv`

We chose to use the athens_data.csv from the Air Quality Monitoring in European Cities dataset available on Kaggle. It presents a detailed investigation on different environnemental, weather and land indices. The kaggle dataset is on three cities, we chose to use the study on Athen's air quality. The duration is approximately from 2020 to 2023, with an hourly temporal resolution and a spatial resolution of 0.005 degrees.

This dataset is multidimensionnal as it has the Date (temporal), the Latitude and Longitude (spatial), and the analytical data from Athen's pollution. The dimension is 1,726,464 rows by 19 collumns which are : 

* **Temporal (`YYYY-MM-DD HH:MM:SS`) and Spatial (`°`) data** : `Date`, `Latitude`, `Longitude`, `station_name`
* **Wind speed (`m/s`)** : `Wind-Speed U` (longitude), `Wind-Speed V` (latitude)
* **Temperature** : `Dewpoint Temp`, `Soil Temp`, `Temp`
* **Precipitation (`mm`) and Humidity (`%`)** : `Total Precipitation`, `Relative Humidity`
* **Plant cover** : `Vegetation H`, `Vegetation L`
* **Concentrations (`µg/m³`)** : `PM10` (Aerosol, max diameter of `10µm`), `PM2.5` (`2.5µm`), `NO2` (Nitrogen dioxide), `O3` (Ozone)
* **Air quality** : `code`, `id`

Printed above are a few information about the dataset and each columns.
Below is a recap Table that contains every Column, its datatype, missing values and percentage, minimum and maximum values if they exist.

| Column | datatype | Missing Values | % missing | Min Value | Max Value |
| :--- | :--- | :---: | :---: | :---: | :---: |
| `Date` | `object` (datetime) | 0 | 0,00 % | `2020-05-01 00:00:00` | `2023-05-29 23:00:00` |
| `Latitude` | `float64` | 0 | 0,00 % | `37.8139` | `38.1500` |
| `Longitude` | `float64` | 0 | 0,00 % | `23.5380` | `23.9500` |
| `station_name` | `object` (string) | 0 | 0,00 % | - | - |
| `Wind-Speed (U)` | `float64` | 8 573 | 0,50 % | `-5.6148` | `7.1708` |
| `Wind-Speed (V)` | `float64` | 8 573 | 0,50 % | `-9.5503` | `10.0797` |
| `Dewpoint Temp` | `float64` | 8 573 | 0,50 % | `-10.9031` | `23.2127` |
| `Soil Temp` | `float64` | 8 573 | 0,50 % | `1.9578` | `37.0462` |
| `Total Percipitation` | `float64` | 8 573 | 0,50 % | `-3.47e-18` *(≈ 0)* | `0.0395` |
| `Vegitation (High)` | `float64` | 8 573 | 0,50 % | `0.4205` | `2.0112` |
| `Vegitation (Low)` | `float64` | 8 573 | 0,50 % | `0.6636` | `2.3057` |
| `Temp` | `float64` | 7 565 | 0,44 % | `-3.2855` | `42.6083` |
| `Relative Humidity` | `float64` | 8 573 | 0,50 % | `8.9178` | `99.9366` |
| `PM10` | `float64` | 0 | 0,00 % | `0.0000` | `1499.6870` |
| `PM2.5` | `float64` | 0 | 0,00 % | `0.0000` | `1748.7150` |
| `NO2` | `float64` | 143 293 | 8,30 % | `0.2716` | `312.0000` |
| `O3` | `float64` | 148 064 | 8,58 % | `0.0091` | `303.0000` |
| `code` | `object` (string) | 26 976 | 1,56 % | - | - |
| `id` | `int64` | 0 | 0,00 % | `9` | `101` |

# Data Quality :
* **Complete Spatial and Temporal Data** : `Date`, `Latitude`, `Longitude`, `station_name` and `id` have no missing values.
* **Missing values in Gas Concentration** : `NO2` (8,30 %) and `O3` (8,58 %) are missing the most values, probably because not all stations measure these.
* **Weather missing values** : The majority of weather indices are missing 8 573 values (0,50 %).
* **Anomalies and notable details** :
  * The columns `Total Percipitation`, `Vegitation (High)` and `Vegitation (Low)` all contain spelling mistakes.
  * `Total Percipitation` has a negative minimum (`-3.47e-18`), probably due to noise rounded to zero.
  * Some very high maximal values appear for `PM10` (1500 µg/m³) and `PM2.5` (1749 µg/m³), it may represent a spike in pollution or outliers.

In [17]:
# Function to load and preprocess the dataset
# Input: file_path (str) - path to the CSV file
# Output: df (pd.DataFrame) - cleaned dataframe with parsed Date column
def load_and_clean_data(file_path):
    # Load CSV file into a pandas DataFrame
    df = pd.read_csv(file_path)
    # Convert Date column to datetime format for temporal operations
    df["Date"] = pd.to_datetime(df["Date"])
    # Fix tiny floating-point rounding negatives in precipitation
    df["Total Percipitation"] = df["Total Percipitation"].clip(lower=0)
    return df


# Function to compute basic exploratory statistics (Task 1)
# Input: df (pd.DataFrame) - raw or cleaned dataframe
# Output: summary_df (pd.DataFrame) - table with dtypes, missing values, min, and max
def explore_dataset(df):
    # Build a summary table per column
    summary_df = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_values": df.isnull().sum(),
        "min": df.min(numeric_only=False),
        "max": df.max(numeric_only=False)
    })
    return summary_df

In [18]:
# Query 1 : Grouping query on PM2.5 and NO2 levels by station
# Input: df (pd.DataFrame) - cleaned dataframe
# Output: station_stats (pd.DataFrame) - mean, max, and exceedance hours per station
def query_grouping_station_pollution(df: pd.DataFrame) -> pd.DataFrame:
    # Compute mean, max, and count of hours where PM2.5 > 25 µg/m3 per station
    station_stats = (
        df.groupby("station_name")
        .agg(
            pm25_mean=("PM2.5", "mean"),
            pm25_max=("PM2.5", "max"),
            no2_mean=("NO2", "mean"),
            pm25_exceedance_hours=("PM2.5", lambda s: (s > 25).sum())
        )
        .sort_values(by="pm25_mean", ascending=False)
        .reset_index()
    )
    return station_stats

In [ ]:
if __name__ == "__main__":
    # Step 1: Data collection and exploration
    file_path = "athens_data.csv"
    df = load_and_clean_data(file_path)
    summary_table = explore_dataset(df)

    # Step 2: Information building (4 queries)
    q1_station_stats = query_grouping_station_pollution(df)
    q2_association_rules = query_frequent_patterns(df)
    q3a_forecast_df, q3a_metrics = query_temporal_forecasting(df)
    q3b_spatial_clusters = query_spatial_clustering(df, n_clusters=3)

    # Step 3: Launch visualization dashboard (Dash)
    # app = build_dashboard(q1_station_stats, q2_association_rules, q3a_forecast_df, q3b_spatial_clusters)
    # app.run(debug=False)